# Corporate Intelligence and Strategic Brochure Studio
## An Advanced Gradio Blocks Multi-Page Analysis Engine

### Architectural Overview
Standard AI brochure generators rely on single landing pages, missing critical business data located on secondary sub-pages such as pricing tiers, leadership bios, technical specifications, and career postings.

This project implements a complete Corporate Intelligence Studio built on Gradio Blocks (`gr.Blocks`). The application executes a multi-stage autonomous pipeline:
1. Crawls and parses the primary target website.
2. Uses one-shot JSON extraction to discover and classify strategic internal sub-pages.
3. Recursively aggregates multi-page content with real-time UI progress updates via `gr.Progress`.
4. Synthesizes a structured intelligence brief customized by target audience and tone.
5. Streams the report token-by-token into a multi-tab interface and generates an exportable Markdown artifact.

In [ ]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_links, fetch_website_contents

## Client Initialization and Authentication Routing

This engine supports both the standard paid OpenAI client and free-tier alternatives:
- Standard OpenAI endpoint (`gpt-4.1-mini`) for maintainers and repository reviewers.
- Google Gemini direct endpoint (`gemini-3.1-flash-lite`) providing 500 free requests per day.
- OpenRouter free endpoint (`openrouter/free`) for multi-provider routing.

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Configuration for model gateway
if google_api_key:
    os.environ["GEMINI_API_KEY"] = google_api_key

# Default client setup with universal fallbacks
if openai_api_key and openai_api_key.startswith("sk-proj-"):
    print("Using standard OpenAI production client.")
    client = OpenAI(api_key=openai_api_key)
    DEFAULT_MODEL = "gpt-4.1-mini"
elif google_api_key:
    print("Configured Google Gemini OpenAI-compatible client (Free Tier).")
    client = OpenAI(
        api_key=google_api_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
    DEFAULT_MODEL = "gemini-3.1-flash-lite"
elif openrouter_api_key:
    print("Configured OpenRouter universal client (Free Tier).")
    client = OpenAI(
        api_key=openrouter_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    DEFAULT_MODEL = "openrouter/free"
else:
    print("Warning: No active API key detected in environment.")
    client = OpenAI()
    DEFAULT_MODEL = "gpt-4.1-mini"

## Autonomous Strategic Sub-Page Discovery

The landing page alone typically lacks depth. To discover relevant corporate intelligence, the system extracts all hyperlinks on the domain and queries the LLM with a strict JSON system prompt.

The prompt requires the model to return a structured JSON list classifying relevant URLs (such as About, Product, Pricing, and Careers pages) while filtering out privacy policies, terms of service, and tracking scripts. Robust string boundary slicing (`find('{')` and `rfind('}')`) prevents JSON parsing failures caused by markdown wrappers.

In [ ]:
link_system_prompt = """
You are a corporate intelligence analyst. Given a list of internal hyperlinks found on a corporate website,
identify up to 4 the most strategically valuable links for corporate diligence (e.g., About, Products, Pricing, Technology, Careers).
Ignore terms of service, privacy policies, login pages, and third-party social media.

Respond STRICTLY in JSON format matching this schema:
{
    "links": [
        {"type": "About Page", "url": "https://example.com/about"},
        {"type": "Pricing Page", "url": "https://example.com/pricing"}
    ]
}
"""

def discover_strategic_links(target_url, max_links=3):
    """
    Extracts hyperlinks from a target URL and classifies the most valuable pages in JSON format.
    """
    try:
        raw_links = fetch_website_links(target_url)
    except Exception as e:
        return []

    if not raw_links:
        return []

    user_prompt = f"Target Website: {target_url}\nAvailable Links:\n" + "\n".join(raw_links[:100])

    try:
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[
                {"role": "system", "content": link_system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        
        # Robust string slicing to guarantee clean JSON extraction
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1:
            clean_json = content[start:end+1]
            parsed = json.loads(clean_json)
            return parsed.get("links", [])[:max_links]
    except Exception as err:
        print(f"Strategic link discovery failed: {err}")
        return []
    
    return []

## Multi-Stage Recursive Web Scraper

Once the discovery engine selects the strategic sub-pages, the recursive aggregation pipeline executes:
1. Scrapes the primary landing page content.
2. Iterates over each selected sub-page to extract its text.
3. Truncates and normalizes character counts to fit safely inside context windows.
4. Updates Gradio's native `gr.Progress` tracker so the user can observe crawling milestones in real time.

In [ ]:
def aggregate_corporate_context(company_name, target_url, max_subpages, progress=gr.Progress()):
    """
    Fetches the landing page and all discovered strategic sub-pages with real-time UI progress updates.
    """
    telemetry_log = []
    
    progress(0.1, desc="Connecting to root landing page...")
    landing_text = fetch_website_contents(target_url)
    telemetry_log.append(f"Landing Page ({target_url}): Scraped {len(landing_text)} characters.")
    
    progress(0.3, desc="Discovering strategic sub-pages via LLM classification...")
    discovered = discover_strategic_links(target_url, max_links=max_subpages)
    
    aggregated_context = f"# Intelligence Dossier: {company_name}\n\n## Landing Page Data:\n{landing_text[:4000]}\n\n"
    
    if discovered:
        telemetry_log.append(f"Discovered {len(discovered)} strategic sub-pages for deep crawling.")
        step_increment = 0.5 / max(len(discovered), 1)
        
        for idx, item in enumerate(discovered, start=1):
            page_type = item.get("type", "Sub-Page")
            page_url = item.get("url", "")
            progress(0.4 + (idx * step_increment), desc=f"Scraping [{page_type}]: {page_url}")
            
            try:
                sub_text = fetch_website_contents(page_url)
                aggregated_context += f"## Strategic Section [{page_type}] ({page_url}):\n{sub_text[:3000]}\n\n"
                telemetry_log.append(f"Success: [{page_type}] - {len(sub_text)} characters fetched.")
            except Exception as e:
                telemetry_log.append(f"Failed to scrape [{page_type}] at {page_url}: {str(e)}")
    else:
        telemetry_log.append("No strategic sub-pages selected; proceeding with landing page data only.")
        
    progress(0.9, desc="Context aggregation complete. Preparing synthesis...")
    return aggregated_context, "\n".join(telemetry_log)

## Dynamic Audience and Tone Prompt Construction

To deliver commercial value, the model adapts its synthesis parameters based on the selected audience:
- **Institutional Investors:** Emphasizes unit economics, market positioning, moat, leadership credentials, and enterprise scaling.
- **Enterprise Clients:** Focuses on product capabilities, security, ROI, integration flexibility, and customer support.
- **Prospective Talent:** Focuses on mission, engineering culture, technical stack, organizational values, and career growth.
- **Technical Partners:** Focuses on architecture, API capabilities, performance benchmarks, and ecosystem integration.

In [ ]:
def construct_synthesis_prompt(company_name, audience, tone):
    system_instruction = f"""
You are an executive intelligence consultant creating an authoritative corporate brochure and strategic briefing for {company_name}.
Target Audience: {audience}
Executive Tone: {tone}

Formatting Guidelines:
- Respond in clean, polished Markdown without using code blocks.
- Include a Header with the Company Name, Audience, and Executive Summary.
- Organize the report into structured sections:
  1. Executive Summary and Value Proposition
  2. Core Offerings and Technical Architecture
  3. Strategic Differentiation and Market Moat
  4. Audience-Specific Deep Dive (tailored directly to {audience})
  5. Strategic Outlook and Call to Action
"""
    return system_instruction

def generate_studio_brief(company_name, target_url, audience, tone, max_subpages, progress=gr.Progress()):
    """
    Main orchestration generator. Coordinates scraping, telemetry logging, and token streaming.
    """
    if not company_name or not target_url:
        yield "Error: Please provide both a Company Name and a valid Website URL.", "Input validation failed."
        return

    # Phase 1: Aggregate multi-page context
    aggregated_context, telemetry = aggregate_corporate_context(
        company_name, target_url, max_subpages, progress=progress
    )
    
    # Phase 2: Construct prompts
    system_prompt = construct_synthesis_prompt(company_name, audience, tone)
    user_prompt = f"Analyze the following aggregated corporate intelligence and generate the strategic brief:\n\n{aggregated_context[:8000]}"
    
    progress(0.95, desc="Streaming executive intelligence report...")
    
    # Phase 3: Stream completion
    try:
        stream = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            stream=True
        )
        
        dossier_text = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                dossier_text += chunk.choices[0].delta.content
                yield dossier_text, telemetry
                
    except Exception as err:
        yield f"API Execution Error: {str(err)}", telemetry

## Export Artifact Generation

This utility packages the generated Markdown intelligence report into a downloadable file. When the user clicks the Export button, the application writes the buffer to a local file and presents a direct browser download link.

In [ ]:
def export_dossier_file(content, company_name):
    """
    Writes the generated markdown report to a local file for direct download.
    """
    if not content or content.startswith("Error:"):
        return None
    
    clean_name = "".join(c for c in company_name if c.isalnum() or c in (" ", "_", "-")).rstrip()
    filename = f"{clean_name.replace(' ', '_').lower()}_executive_brief.md"
    filepath = os.path.join(os.getcwd(), filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
        
    return filepath

## Gradio Blocks Interface Implementation

We construct the user interface using `gr.Blocks()`:
- **Left Column:** Controls for company inputs, audience targeting, tone selection, and an accordion for deep crawling depth.
- **Right Column:** A dual-tab output workspace containing:
  - **Tab 1 (Executive Briefing):** Real-time streaming Markdown report and artifact download button.
  - **Tab 2 (Discovery Telemetry):** Live logs showing discovered URLs and character extraction metrics.

In [ ]:
with gr.Blocks(title="Corporate Intelligence & Strategic Brochure Studio") as demo:
    gr.Markdown("# Corporate Intelligence and Strategic Brochure Studio")
    gr.Markdown("Autonomous multi-page discovery, audience targeting, and real-time streaming intelligence.")
    
    with gr.Row():
        with gr.Column(scale=4):
            company_input = gr.Textbox(
                label="Company Name", 
                placeholder="e.g., Anthropic, Databricks, Stripe",
                lines=1
            )
            url_input = gr.Textbox(
                label="Website URL", 
                placeholder="https://example.com",
                lines=1
            )
            
            audience_selector = gr.Dropdown(
                label="Target Audience",
                choices=[
                    "Institutional Investors & VCs",
                    "Enterprise Commercial Clients",
                    "Prospective Talent & Engineers",
                    "Strategic Technology Partners"
                ],
                value="Institutional Investors & VCs"
            )
            
            tone_selector = gr.Radio(
                label="Executive Tone",
                choices=["Rigorous Analytical", "Visionary Strategic", "Technical Deep Dive"],
                value="Rigorous Analytical"
            )
            
            with gr.Accordion("Deep Crawling and Optimization Settings", open=False):
                subpage_slider = gr.Slider(
                    label="Maximum Strategic Sub-Pages to Crawl",
                    minimum=1,
                    maximum=4,
                    value=3,
                    step=1,
                    info="Controls how many secondary links (About, Pricing, Careers) are fetched."
                )
                
            generate_button = gr.Button("Generate Strategic Dossier", variant="primary")
            
        with gr.Column(scale=6):
            with gr.Tabs():
                with gr.TabItem("Executive Briefing"):
                    brief_output = gr.Markdown(label="Synthesized Brief")
                    export_button = gr.Button("Prepare Markdown Export", variant="secondary")
                    file_download = gr.File(label="Download Dossier File", visible=False)
                    
                with gr.TabItem("Discovery Telemetry Log"):
                    telemetry_output = gr.Textbox(
                        label="Scraping & Link Discovery Log",
                        lines=14,
                        interactive=False
                    )

    # Event binding for streaming generation
    generate_button.click(
        fn=generate_studio_brief,
        inputs=[company_input, url_input, audience_selector, tone_selector, subpage_slider],
        outputs=[brief_output, telemetry_output]
    )
    
    # Event binding for file export
    def trigger_export(content, company):
        path = export_dossier_file(content, company)
        if path:
            return gr.update(value=path, visible=True)
        return gr.update(visible=False)

    export_button.click(
        fn=trigger_export,
        inputs=[brief_output, company_input],
        outputs=[file_download]
    )

demo.launch(inbrowser=True)